In [77]:
import os
import json
import sys
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
from collections import defaultdict

# Configuration
LANGCHAIN_DIR = Path.cwd()
# Use JSON files for full database provenance (PMCID, text_element_id)
INPUT_JSON_DIR = LANGCHAIN_DIR / "test_results_50_docs" / "umls_entities"
# Legacy TXT files (less provenance data)
INPUT_TXT_DIR = LANGCHAIN_DIR / "test_results_50_docs" / "relevant_texts"
OUTPUT_DIR = LANGCHAIN_DIR / "summarization_results"
SUMMARIES_DIR = OUTPUT_DIR / "summaries"
RULES_DIR = OUTPUT_DIR / "rules"
AUDIT_DIR = OUTPUT_DIR / "audit_trails"

# Create output directories
for directory in [OUTPUT_DIR, SUMMARIES_DIR, RULES_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Input JSON (with DB provenance): {INPUT_JSON_DIR}")
print(f"Input TXT (legacy):              {INPUT_TXT_DIR}")
print(f"Output directory:                {OUTPUT_DIR}")
print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Input JSON (with DB provenance): /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/test_results_50_docs/umls_entities
Input TXT (legacy):              /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/test_results_50_docs/relevant_texts
Output directory:                /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/summarization_results

Started at: 2026-01-21 17:43:51


In [78]:
# !pip install langchain langchain-openai langchain-core python-dotenv tiktoken

In [79]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Load environment variables (API keys)
load_dotenv()

# Verify API key is set
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  Warning: OPENAI_API_KEY not found in environment")
    print("Please set it in .env file or export OPENAI_API_KEY=your_key")
else:
    print("✅ OpenAI API key loaded successfully")

✅ OpenAI API key loaded successfully


### Let's load all json files from the INPUT_JSON_DIR directory.

In [80]:
def load_json_files_with_provenance(input_dir: Path, limit: Optional[int] = None) -> List[Dict]:
    """
    Load UMLS entity JSON files with FULL DATABASE PROVENANCE.
    
    Each sentence includes:
    - pmcid: Links to documents.pmcid in database
    - text_element_id: Links to text_elements.id in database
    - section: Section context from text_elements.path_string
    - entity_text, start_char, end_char: Entity position
    - umls_score: Entity linking confidence
    
    This provides complete traceability back to the database schema.
    
    Args:
        input_dir: Directory containing JSON files
        limit: Optional limit on number of files to load
    
    Returns:
        List of dicts with full provenance metadata
    """
    json_files = sorted(input_dir.glob("*.json"))
    
    if limit:
        json_files = json_files[:limit]
    
    loaded_files = []
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata
            cui = data.get('umls_cui', '')
            concept_name = data.get('canonical_name', '')
            entity_label = data.get('entity_label', '')
            
            # Extract sentences with full provenance
            sentences_with_provenance = []
            for sent_data in data.get('sentences', []):
                sentences_with_provenance.append({
                    'pmcid': sent_data.get('pmcid'),  # Links to documents.pmcid
                    'text_element_id': sent_data.get('text_element_id'),  # Links to text_elements.id
                    'sentence': sent_data.get('sentence'),
                    'section': sent_data.get('section'),
                    'entity_text': sent_data.get('entity_text'),
                    'start_char': sent_data.get('start_char'),
                    'end_char': sent_data.get('end_char'),
                    'umls_score': sent_data.get('umls_score')
                })
            
            # Group by PMCID for statistics
            pmcids = set(s['pmcid'] for s in sentences_with_provenance if s['pmcid'])
            
            loaded_files.append({
                'cui': cui,
                'concept_name': concept_name,
                'entity_label': entity_label,
                'filename': json_file.name,
                'filepath': str(json_file),
                'sentences_with_provenance': sentences_with_provenance,
                # Simple sentence list for compatibility
                'sentences': [s['sentence'] for s in sentences_with_provenance],
                'num_sentences': len(sentences_with_provenance),
                'num_documents': len(pmcids),
                'pmcids': list(pmcids),
                'metadata': {
                    'umls_cui': cui,
                    'canonical_name': concept_name,
                    'entity_label': entity_label,
                    'total_occurrences': data.get('total_occurrences', 0),
                    'unique_entity_texts': data.get('unique_entity_texts', [])
                }
            })
        
        except Exception as e:
            print(f"Warning: Could not load {json_file.name}: {e}")
    
    return loaded_files


# Load JSON files with full provenance
print("Loading JSON files with database provenance...")
json_files = load_json_files_with_provenance(INPUT_JSON_DIR, limit=10)

print(f"\nLoaded {len(json_files)} concept files with DB provenance")
print("\nSample file structure:")
if json_files:
    sample = json_files[0]
    print(f"  CUI: {sample['cui']}")
    print(f"  Concept: {sample['concept_name']}")
    print(f"  Sentences: {sample['num_sentences']}")
    print(f"  Documents (PMCIDs): {sample['num_documents']}")
    if sample['sentences_with_provenance']:
        sent = sample['sentences_with_provenance'][0]
        print(f"\n  Sample sentence provenance:")
        print(f"    PMCID: {sent['pmcid']} -> documents.pmcid")
        print(f"    text_element_id: {sent['text_element_id']} -> text_elements.id")
        print(f"    section: {sent['section']}")
        print(f"    umls_score: {sent['umls_score']}")

Loading JSON files with database provenance...

Loaded 10 concept files with DB provenance

Sample file structure:
  CUI: C0000368
  Concept: 3,3'-Diaminobenzidine
  Sentences: 1
  Documents (PMCIDs): 1

  Sample sentence provenance:
    PMCID: PMC11503264 -> documents.pmcid
    text_element_id: 446 -> text_elements.id
    section: 2. Materials and Methods
    umls_score: 0.7855776


### Get the summary for each file.

In [81]:
# =============================================================================
# AUDITABLE MAP/REDUCE SYSTEM PROMPTS
# =============================================================================
# This system maintains full traceability from final summary -> chunks -> source sentences -> database
#
# Provenance Chain:
#   Final Summary [claim] -> Chunk ID -> Sentence IDs -> PMCID + text_element_id (database)
#
# Output Format: Structured JSON blocks enable machine parsing for audit validation
# =============================================================================

MAP_PROMPT = """<Role>You are a Medical Evidence Analyst processing a chunk of histopathology literature.</Role>

<Context>
Concept: {concept_name}
Chunk ID: {chunk_id}
Source Sentences (each tagged with [SentenceID|PMCID|TextElementID]):
{text}
</Context>

<Task>
Analyze this chunk and produce a STRUCTURED AUDITABLE SUMMARY with full provenance tracking.

CRITICAL AUDIT REQUIREMENTS:
1. Every factual claim MUST cite specific sentence IDs from the input
2. Use the exact citation format: [S1|PMC123|te456] where available
3. Never synthesize or infer claims without direct textual support
4. If information conflicts, cite all sources and note the conflict
</Task>

<OutputFormat>
Return your analysis in this EXACT structure:

```json
{{
  "chunk_id": "{chunk_id}",
  "findings": [
    {{
      "category": "diagnostic|histopathological|treatment|prognostic|risk_factor",
      "claim": "<factual statement>",
      "evidence": ["S1|PMC123|te456", "S3|PMC123|te458"],
      "confidence": "high|medium|low",
      "verbatim_support": "<key quote from source>"
    }}
  ],
  "summary_text": "<narrative summary with inline citations [S1|PMC...]>",
  "audit_metadata": {{
    "sentences_analyzed": <count>,
    "sentences_cited": [<list of cited sentence IDs>],
    "pmcids_referenced": [<list of PMCIDs>],
    "uncited_sentences": [<list of uncited sentence IDs>]
  }}
}}
```
</OutputFormat>

Structured Analysis:"""

In [82]:
REDUCE_PROMPT = """
<Role>
You are a Lead Pathologist synthesizing chunk-level analyses into a Master Clinical Brief with FULL AUDIT TRAIL.
</Role>

<Context>
Concept: {concept_name}
Total Chunks: {num_chunks}

Chunk Analyses (JSON format with provenance):
{summaries}
</Context>

<Task>
Consolidate all chunk analyses into a unified report while PRESERVING THE COMPLETE AUDIT CHAIN.

CRITICAL AUDIT REQUIREMENTS:
1. EVERY claim must trace back to: Chunk ID -> Sentence ID -> PMCID -> text_element_id
2. Use format: [Chunk:C1, Evidence:S2|PMC123|te456]
3. When multiple chunks support a finding, list ALL sources
4. Flag any conflicting evidence with all sources cited
5. Do NOT add information not present in the chunk analyses
</Task>

<OutputFormat>
```json
{{
  "concept": "{concept_name}",
  "sections": {{
    "clinical_significance": {{
      "findings": [
        {{
          "claim": "<statement>",
          "sources": [
            {{"chunk": "C1", "sentences": ["S1|PMC...|te..."], "verbatim": "<quote>"}}
          ],
          "strength": "strong|moderate|weak"
        }}
      ]
    }},
    "histopathological_features": {{ ... }},
    "management_outcomes": {{ ... }},
    "risk_factors_associations": {{ ... }}
  }},
  "narrative_summary": "<readable summary with inline citations [C1:S2|PMC...]>",
  "audit_trail": {{
    "chunks_processed": <count>,
    "total_sentences_cited": <count>,
    "unique_pmcids": [<list>],
    "unique_text_element_ids": [<list>],
    "evidence_conflicts": [
      {{"topic": "...", "conflicting_sources": [...]}}
    ]
  }}
}}
```
</OutputFormat>

Master Auditable Summary:"""

In [83]:
RULE_EXTRACTION_PROMPT = """
<Role>
You are a Medical Knowledge Engineer. Extract structured IF-THEN rules with COMPLETE PROVENANCE from the Auditable Summary.
</Role>

<Input>
Concept: {concept_name}
Auditable Summary (JSON with full provenance):
{summary}
</Input>

<Task>
Extract actionable clinical rules, each with FULL TRACEABILITY back to source documents.

AUDIT REQUIREMENTS:
1. Each rule must cite specific evidence from the summary
2. Trace back through: Rule -> Summary Claim -> Chunk -> Sentence -> PMCID/text_element_id
3. Include the database reference for each supporting sentence
</Task>

<OutputFormat>
```json
{{
  "rules": [
    {{
      "rule_id": "R1",
      "type": "Diagnostic|Prognostic|Management",
      "condition": "IF <observation>",
      "action": "THEN <conclusion>",
      "confidence": "High|Medium|Low",
      "evidence_chain": [
        {{
          "chunk_id": "C1",
          "sentence_id": "S2",
          "pmcid": "PMC123456",
          "text_element_id": 789,
          "verbatim": "<supporting quote>"
        }}
      ],
      "contraindications": ["<any noted exceptions with sources>"]
    }}
  ],
  "audit_summary": {{
    "total_rules": <count>,
    "rules_by_type": {{"Diagnostic": N, "Prognostic": N, "Management": N}},
    "pmcids_supporting_rules": [<list>],
    "average_evidence_per_rule": <float>
  }}
}}
```
</OutputFormat>

Extracted Rules with Provenance:"""

#### Generate template from prompt for summarization:

In [84]:
MAPPING_TEMPLATE = ChatPromptTemplate(
    [('system', MAP_PROMPT)]
)

REDUCE_TEMPLATE = ChatPromptTemplate(
    [('system', REDUCE_PROMPT)]
)

RULE_EXTRACTION_TEMPLATE = ChatPromptTemplate(
    [('system', RULE_EXTRACTION_PROMPT)]
)

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Literal

# =============================================================================
# MAP SCHEMA - matches MAP_PROMPT output format
# =============================================================================
class Finding(BaseModel):
    category: str = Field(description="diagnostic|histopathological|treatment|prognostic|risk_factor")
    claim: str = Field(description="The factual medical statement")
    evidence: List[str] = Field(description="List of citation IDs e.g. S1|PMC123|te456")
    confidence: Literal["high", "medium", "low"]
    verbatim_support: str = Field(description="Exact quote from the source text")

class AuditMetadata(BaseModel):
    sentences_analyzed: int
    sentences_cited: List[str]
    pmcids_referenced: List[str]
    uncited_sentences: List[str] = Field(description="List of sentence IDs not used in findings")

class AuditableSummary(BaseModel):
    chunk_id: str
    findings: List[Finding]
    summary_text: str
    audit_metadata: AuditMetadata


# =============================================================================
# REDUCE SCHEMA - matches REDUCE_PROMPT output format
# =============================================================================
class SourceReference(BaseModel):
    chunk: str
    sentences: List[str]
    verbatim: str

class SectionFinding(BaseModel):
    claim: str
    sources: List[SourceReference]
    strength: Literal["strong", "moderate", "weak"]

class Section(BaseModel):
    findings: List[SectionFinding]

# NEW: Explicitly define the sections instead of using a Dict
class SectionsContainer(BaseModel):
    clinical_significance: Section
    histopathological_features: Section
    management_outcomes: Section
    risk_factors_associations: Section

class EvidenceConflict(BaseModel):
    topic: str
    conflicting_sources: List[str]

class ReduceAuditTrail(BaseModel):
    chunks_processed: int
    total_sentences_cited: int
    unique_pmcids: List[str]
    unique_text_element_ids: List[int]
    evidence_conflicts: List[EvidenceConflict] 

class ConsolidatedSummary(BaseModel):
    concept: str
    sections: SectionsContainer
    narrative_summary: str
    audit_trail: ReduceAuditTrail


# =============================================================================
# RULE EXTRACTION SCHEMA - matches RULE_EXTRACTION_PROMPT output format
# =============================================================================
class EvidenceChainItem(BaseModel):
    chunk_id: str
    sentence_id: str
    pmcid: str
    text_element_id: int
    verbatim: str

class Rule(BaseModel):
    rule_id: str
    type: Literal["Diagnostic", "Prognostic", "Management"]
    condition: str = Field(description="IF <observation>")
    action: str = Field(description="THEN <conclusion>")
    confidence: Literal["High", "Medium", "Low"]
    evidence_chain: List[EvidenceChainItem]
    contraindications: List[str]

class RuleCounts(BaseModel):
    Diagnostic: int = Field(description="Count of Diagnostic rules")
    Prognostic: int = Field(description="Count of Prognostic rules")
    Management: int = Field(description="Count of Management rules")

class RuleAuditSummary(BaseModel):
    total_rules: int
    rules_by_type: RuleCounts
    pmcids_supporting_rules: List[str]
    average_evidence_per_rule: float

class ExtractedRules(BaseModel):
    rules: List[Rule]
    audit_summary: RuleAuditSummary


# =============================================================================
# LLM and Chain Setup
# =============================================================================
cheap_map_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1).with_retry(
    stop_after_attempt=3
)

smart_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_retry(
    stop_after_attempt=3
)

map_chain = MAPPING_TEMPLATE | cheap_map_llm.with_structured_output(AuditableSummary, strict=True).with_retry
reduce_chain = REDUCE_TEMPLATE | smart_llm.with_structured_output(ConsolidatedSummary, strict=True)
rule_chain = RULE_EXTRACTION_TEMPLATE | smart_llm.with_structured_output(ExtractedRules, strict=True)

In [ ]:
import tiktoken

class TokenTracker:
    def __init__(self):
        self.totals = {
            "gpt-4o-mini": {"input": 0, "output": 0},
            "gpt-4o": {"input": 0, "output": 0}
        }
        # Prices per 1M tokens (as of early 2026 - verify with latest docs)
        self.prices = {
            "gpt-4o-mini": {"input": 0.15, "output": 0.60},
            "gpt-4o": {"input": 2.50, "output": 10.00}
        }

    def add_usage(self, model: str, input_str: str, output_obj: any):
        enc = tiktoken.encoding_for_model(model)
        
        # Count input tokens
        self.totals[model]["input"] += len(enc.encode(input_str))
        
        # Count output tokens (convert Pydantic object back to string)
        output_str = json.dumps(output_obj.model_dump())
        self.totals[model]["output"] += len(enc.encode(output_str))

    def calculate_total_cost(self):
        total_cost = 0
        for model, usage in self.totals.items():
            in_cost = (usage["input"] / 1_000_000) * self.prices[model]["input"]
            out_cost = (usage["output"] / 1_000_000) * self.prices[model]["output"]
            total_cost += (in_cost + out_cost)
        return round(total_cost, 4)

In [ ]:
def minify_summaries(summaries: List[BaseModel]) -> str:
    """
    Converts Pydantic objects to a compact JSON string to save tokens.
    Removes all whitespace and newlines.
    """
    # model_dump(exclude_none=True) prevents sending empty fields
    data = [s.model_dump(exclude_none=True) for s in summaries]
    return json.dumps(data, separators=(',', ':'))

In [86]:
def format_sentences_for_llm(chunk: list) -> str:
    """
    Formats a list of sentence dictionaries into a structured string 
    for the LLM to easily reference citation IDs.
    """
    formatted_lines = []
    
    for i, item in enumerate(chunk):
        pmcid = item.get('pmcid', 'UNKNOWN')
        te_id = item.get('text_element_id', '0')
        sentence_text = item.get('sentence', '').strip()
        
        # Create a unique Citation ID for this specific sentence
        # Format: S{index}|{PMCID}|{ElementID}
        citation_id = f"S{i+1}|{pmcid}|{te_id}"
        
        # Format the line for the prompt
        formatted_lines.append(f"[{citation_id}] {sentence_text}")
    
    return "\n".join(formatted_lines)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def process_document_hierarchical(file_data: Dict, map_chain, reduce_chain, rule_chain) -> Dict:
    """
    Process document using hierarchical Map-Reduce to avoid truncation.
    Returns Pydantic objects with full provenance tracking.
    """
    concept_name = file_data['concept_name']
    cui = file_data['cui']
    sentences_with_metadata = file_data['sentences_with_provenance']
    
    try:
        # 2. MAP STEP: Process sentences in chunks of 10
        chunk_size = 10
        sentence_chunks = [sentences_with_metadata[i:i + chunk_size] for i in range(0, len(sentences_with_metadata), chunk_size)]
        
        # Parallelize the Map calls for speed
        map_inputs = [
            {
                "concept_name": concept_name, 
                "chunk_id": f"C{i+1}", 
                "text": format_sentences_for_llm(chunk)
            }
            for i, chunk in enumerate(sentence_chunks)
        ]
        current_summaries: List[AuditableSummary] = map_chain.batch(map_inputs)
        
        
        # 3. REDUCE STEP: Recursively collapse summaries
        # Group summaries into batches of 10 and reduce them until only 1 remains
        if len(current_summaries) == 1:
            master_summary: ConsolidatedSummary = reduce_chain.invoke({
                "concept_name": concept_name,
                "num_chunks": 1,
                "summaries": minify_summaries(current_summaries)
            })
        else:
            while len(current_summaries) > 1:
                summary_groups = [current_summaries[i:i + 10] for i in range(0, len(current_summaries), 10)]
                
                reduce_inputs = [
                    {
                        "concept_name": concept_name,
                        "num_chunks": len(group),
                        "summaries": minify_summaries(group)
                    }
                    for group in summary_groups
                ]

                next_level_summaries: List[ConsolidatedSummary] = reduce_chain.batch(reduce_inputs)
                current_summaries = next_level_summaries

            master_summary: ConsolidatedSummary = current_summaries[0]
        
        # 4. RULE EXTRACTION: Run on the final Master Summary object
        final_rules: ExtractedRules = rule_chain.invoke({
            "concept_name": concept_name,
            "summary": minify_summaries([master_summary])
        })
        
        return {
            'status': 'success',
            'cui': cui,
            'concept_name': concept_name,
            'summary': master_summary.narrative_summary,
            'rules': [rule.model_dump() for rule in final_rules.rules],
            'audit_trail': {
                'master_summary': master_summary.model_dump(),
                'rules_provenance': final_rules.model_dump()
            }
        }

    except Exception as e:
        return {
            'status': 'error',
            'cui': cui,
            'concept_name': concept_name,
            'error': str(e)
        }

## Start processing.

In [88]:
def get_result_path(cui: str) -> Path:
    """Get the path where a result would be saved."""
    return SUMMARIES_DIR / f"{cui}.json"

def load_existing_result(cui: str) -> Optional[Dict]:
    """Load an existing result if it exists."""
    result_path = get_result_path(cui)
    if result_path.exists():
        with open(result_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def save_result(result: Dict) -> None:
    """Save a result to disk."""
    cui = result['cui']
    result_path = get_result_path(cui)
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)


results = []
errors = []
skipped = []

print("="*80)
print("Starting Processing Pipeline")
print("="*80 + "\n")

for i, file_data in enumerate(json_files, 1):
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    print(f"[{i}/{len(json_files)}] {cui} - {concept_name[:50]}...", end=" ")
    
    # Check if already processed
    existing = load_existing_result(cui)
    if existing is not None:
        print("SKIPPED (cached)")
        skipped.append(existing)
        continue
    
    result = process_document_hierarchical(
        file_data, 
        map_chain=map_chain, 
        reduce_chain=reduce_chain, 
        rule_chain=rule_chain
    )
    
    if result['status'] == 'success':
        save_result(result)
        results.append(result)
        print(f"OK ({len(result['summary'])} chars, {len(result['rules'])} rules)")
    else:
        errors.append(result)
        print(f"ERROR: {result['error']}")

print("\n" + "="*80)
print(f"Processing Complete:")
print(f"  New:    {len(results)}")
print(f"  Cached: {len(skipped)}")
print(f"  Errors: {len(errors)}")
print("="*80)

Starting Processing Pipeline

[1/10] C0000368 - 3,3'-Diaminobenzidine... OK (521 chars, 3 rules)
[2/10] C0000723 - Abbreviations... OK (562 chars, 1 rules)
[3/10] C0000726 - Abdomen... OK (1187 chars, 5 rules)
[4/10] C0000737 - Abdominal Pain... 

KeyboardInterrupt: 